In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')
class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"너는 사용자의 질문에 친철하고 정확하게 답변하는 시스템 입니다., Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,            
        )
        return response.choices[0].message.content
llm = OpenAILLM()    

In [13]:
import json
from  textwrap import dedent  #  들여쓰기 indent를 제거
query = '어제 삼성전자 주식종가 를 조회하고 해당 종가와 비슷한종목을 뉴스에서 검색해서 요약하고 오늘날씨에 맞는 외출복을 추천하고 데이트 장소 추천해'
def routerLLM(query):
    prompt = dedent(f"""
    사용자의 질문 의도를 분석하세요.

    질문에 포함된 의도가 여러 개이면
    반드시 모든 의도를 각각 분리하여 출력하세요.

    예를 들어:
    - 주식 + 뉴스 + 날씨
    - 뉴스 + 검색
    - 계산 + 주식

    처럼 복합 질문이면
    반드시 여러 개의 JSON 객체를 배열로 출력해야 합니다.

    question_type 별로 tool_query를 생성하세요.
    tool_query는 반드시 키워드중심으로 생성하세요 llm에 전달하는 문구가 아님을 명심해서 추론을하지말고 검색용 키워드로 매칭해주세요    
    뉴스는 api검색할수 있는 키워드중심으로,
    주식은 종목명만 추출하세요             
    날씨도 검색용 키워드로 추출하세요                               

    지원 가능한 question_type 예시:
    - 날씨
    - 뉴스
    - 주식
    - 검색
    - 계산
    - 지도

    [중요 규칙]
    - 질문에 포함된 모든 의도를 누락 없이 출력
    - 반드시 JSON 배열(Array) 형식으로 출력
    - 하나만 출력 금지
    - 설명문 금지
    - markdown 금지
    - ```json 금지

    [예시 입력]
    어제 삼성전자 종가 알려주고 관련 뉴스와 오늘 날씨도 알려줘

    [예시 출력]
    [
        {{
            "question_type": "주식",
            "tool_query": "삼성전자",
            "reason": "삼성전자 종가 요청"
        }},
        {{
            "question_type": "뉴스",
            "tool_query": "삼성전자 관련 최근 뉴스 검색",
            "reason": "관련 뉴스 요청"
        }},
        {{
            "question_type": "날씨",
            "tool_query": "오늘 날씨 조회",
            "reason": "날씨 요청"
        }}
    ]

    [질문]
    {query}

    [출력]
    반드시 유효한 JSON 배열만 출력하세요.
    """)

    result = llm.generate(prompt)
    json_result = json.loads(result)
    return json_result

In [14]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECETET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(query)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()    
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)        
        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items

In [15]:
# tool excution
tool_results = []
def excute_tools(router_result):
    for row in router_result:
        query_type = row.get('question_type')
        print(f'tool : {query_type}')
        if query_type == '뉴스':        
            query = row.get('tool_query')
            news_result = search_naver_news(row.get('tool_query'))
            tool_results.append({
                'question_type' : '뉴스',
                'query' : query,
                'result' : news_result
            })
        # other tools
    return tool_results            

In [16]:
def make_message(user_query:str, tool_results:list[dict]):
    prompt = f'''
    너는 다양한 도구의 결과를 종합해서 사용자 질문에 답변하는 ai assistant 입니다.

    [사용자질문]
    {user_query}

    [도구실행결과]
    {json.dumps(tool_results, ensure_ascii=False, indent=2)}

    [지침]
    - tool 결과를 기반으로 답변하세요
    - 필요한 경우 여러 tool 결과를 종합하세요
    - 지도 주식 날씨 검색 추천등 다양한 데이터가 포함될수 있습니다.
    - tool 결과내에 있는 내용에 우선순위를 높게해서 해당 데이터기반으로 답변하세요
    '''
    return [
                {"role":"system", "content":"너는 여러 tool결과를 조합해서 최종 답변을생성하는 ai agent 입니다."},
                {"role":"user", "content":prompt}
            ]
    
    
# final llm
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
def final_llm_openai(user_query:str, tool_results:list[dict]):    
    result = client.chat.completions.create(
            model = 'gpt-5.4-nano',
            messages =make_message(user_query,tool_results),
            temperature=0.3            
        )
    return result.choices[0].message.content

### 파이프라인
- 사용자 질문
- 질문의 의도를 분석하는 llm 호출
- 해당 tools을 호출해서 결과 수집
- 최종 llm에게 전달
- 최종 답변 생성

In [17]:
query = '붕괴사고에 대해서'
router_result = routerLLM(query)
tool_results = excute_tools(router_result)
final_result = final_llm_openai(query,tool_results)
print(final_result)

tool : 뉴스
붕괴사고에 대해(최근 보도 기준) 정리하면 아래와 같습니다.

## 1) 서울 서소문 고가차도 붕괴 사고(철도·시설 복구 계획)
- 국토교통부는 **서울 서소문 고가차도 붕괴 사고**와 관련해 **이르면 5월 29일까지 철도 시설을 복구**하는 목표를 밝혔다고 합니다.  
- 또한 **KTX 정상화 때까지 SRT 입석을 확대**하는 등 열차 운행 정상화를 위한 조치도 언급됐습니다.  
  - 출처: 「국토부 “29일까지 철도 복구 목표”…KTX 정상화 때까지 SRT 입석 확대」(2026-05-27)

## 2) 붕괴 원인 파악을 위한 현장 조사(콘크리트 내부 확인)
- 사고 현장에서는 공사 관계자들이 **콘크리트 내부를 확인**하기 위해 **콘크리트 타공 적출물**을 채취·표시하는 작업을 진행했습니다.
- 이번 채취는 **콘크리트가 얼마나 부식됐는지**를 알아보기 위한 것으로, **콘크리트 강도 측정**에도 활용되는 것으로 보도됐습니다.  
  - 출처: 「서소문 고가차도 콘크리트 내부 확인」(2026-05-27)

## 3) 유사 사고 예방을 위한 안전관리(부실시공·붕괴 예방)
- 전남도 관련 보도에서는 **부실시공 및 안전사고 예방**을 위해 위험요인 확인, 작업 전 안전점검, 가설구조물 관리, **추락·붕괴 사고 예방** 등 현장 중심 안전관리 방안을 소개하는 내용이 포함돼 있습니다.  
  - 출처: 「전남도, 부실시공·안전사고 예방 나선다」(2026-05-27)

---

원하시면, “붕괴사고”를 **특정 사건(예: 서소문 고가차도)** 중심으로 더 자세히 요약해드리거나, 아니면 **붕괴사고의 일반적인 원인/예방 체크리스트(시설물 안전점검 관점)** 형태로 정리해드릴까요?


In [18]:
# 1. planner / router : 사용자 요청을 분석하고 무슨작업이 필요한지 결정
# 2. tool excuter : router가 결정한 tool을 실제로 실행
# 3. memory / retrieval : 과거정보와 외부지식을 가져옮
# 4. LLM : 모든 tool결과와 memory를 종합해서 최종답변을 생성
# 5. final response : 사용자에게 보여줄 최종 출력 생성

### 모델을 허깅페이스계열의 모델로 변경

In [19]:
from huggingface_hub import notebook_login

os.environ.pop('HF_TOKEN', None)
notebook_login()

In [20]:
from huggingface_hub import get_token, whoami

HF_TOKEN = get_token()
if not HF_TOKEN:
    raise RuntimeError('Hugging Face 토큰이 저장되지 않았습니다. 로그인 셀을 다시 실행하세요.')

info = whoami(token=HF_TOKEN)
print('logged in user:', info.get('name'))
print('token prefix:', HF_TOKEN[:6] + '***')

logged in user: skn29teacher
token prefix: hf_apS***


### 모델 변경

In [21]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [22]:
def final_llm_qween(user_query:str, tool_results:list[dict]):    
    text = tokenizer.apply_chat_template(
        make_message(user_query,tool_results),
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [23]:
query = '붕괴사고에 대해서'
router_result = routerLLM(query)
tool_results = excute_tools(router_result)  
final_result = final_llm_qween(query,tool_results)
print(final_result)

tool : 뉴스
죄송합니다만, 현재 제공된 정보는 특정 사건이나 사고에 대한 진단 또는 대처 계획에 있어 가장 효과적인 방법을 제공하지 못합니다. 다른 사건이나 사고에 대해 더 자세한 정보를 제공해주시면 감사하겠습니다.
